# 02 · Training the LightGBM baselineRuns the same code path as `python -m src.models.train`, then inspects the result:where the error actually lives, and whether the model beats the naive baselinefor the right reasons.

In [ ]:
import warningswarnings.filterwarnings("ignore")import matplotlib.pyplot as pltimport numpy as npimport pandas as pdfrom src.config import data_dir, load_configfrom src.features.station_clusters import fit_entity_mappingfrom src.ingestion.weather import weather_window_forfrom src.models.dataset import build_dataset, regression_metrics, seasonal_naive_baselinefrom src.models.lightgbm_model import LightGBMDemandModelconfig = load_config()departures = pd.read_parquet(data_dir("cache", "hourly_departures_all.parquet"))departures["hour_ts"] = pd.to_datetime(departures["hour_ts"])stations = pd.read_parquet(data_dir("cache", "station_information.parquet"))mapping = fit_entity_mapping(    stations, config, stations_in_scope=set(departures["station_short_name"]))weather = weather_window_for(departures["hour_ts"], config)dataset = build_dataset(departures, mapping, weather, config)dataset.summary()

The split is **chronological**: validation and test are trailing windows. A randomsplit would let the model see hour `t+1` while predicting hour `t` and producemeaningless scores.

In [ ]:
model = LightGBMDemandModel(dataset.features, config)x_train, y_train = dataset.xy("train")x_val, y_val = dataset.xy("validation")model.fit(x_train, y_train, x_val, y_val)x_test, y_test = dataset.xy("test")predictions = model.predict(x_test)metrics = regression_metrics(y_test, predictions)baseline = seasonal_naive_baseline(dataset.test)print("LightGBM (test):")for key in ["mae", "rmse", "smape", "r2", "mean_actual"]:    print(f"  {key:12s} {metrics[key]:8.3f}")print(f"\nSeasonal-naive baseline MAE: {baseline['baseline_mae']:.3f}")print(f"Improvement over baseline  : {1 - metrics['mae'] / baseline['baseline_mae']:.1%}")

## Where the error livesAn aggregate MAE hides a lot. Two questions matter operationally: does the modelfail at peak hours (when rebalancing decisions are made), and does it fail on thebusiest clusters?

In [ ]:
results = dataset.test[["entity_id", "hour_ts", "departures"]].copy()results["predicted"] = predictionsresults["error"] = results["predicted"] - results["departures"]results["abs_error"] = results["error"].abs()results["hour"] = results["hour_ts"].dt.hourfig, axes = plt.subplots(1, 2, figsize=(14, 4))by_hour = results.groupby("hour").agg(mae=("abs_error", "mean"), actual=("departures", "mean"))axes[0].bar(by_hour.index, by_hour["mae"])axes[0].plot(by_hour.index, by_hour["actual"], color="crimson", marker="o", label="mean actual")axes[0].set(xlabel="hour of day", ylabel="MAE", title="Error concentrates where demand is")axes[0].legend()by_entity = results.groupby("entity_id").agg(mae=("abs_error", "mean"), actual=("departures", "mean"))axes[1].scatter(by_entity["actual"], by_entity["mae"])axes[1].set(xlabel="mean actual demand", ylabel="MAE", title="Error scales with cluster size")plt.tight_layout()print("Relative error is the fairer comparison across clusters:")print((by_entity["mae"] / by_entity["actual"]).describe().round(3))

In [ ]:
importance = model.feature_importance().head(15)fig, ax = plt.subplots(figsize=(8, 5))ax.barh(importance["feature"][::-1], importance["importance"][::-1])ax.set_title("Top 15 features")plt.tight_layout()importance

## Honest reading of these numbers- MAE is reported **against a seasonal-naive baseline**. A forecasting MAE without  one is uninterpretable.- Error scales with cluster size, so the aggregate MAE is dominated by the busiest  clusters. Relative error is the fairer cross-cluster comparison.- The test set is a **single contiguous 14-day window**, so this number carries real  variance. Rolling-origin backtesting is on the roadmap.- Training covers **Apr–Jun only**. `season` is nearly constant, and winter  performance is unknown.To register this model and run champion/challenger promotion, use the CLI ratherthan this notebook — it logs to MLflow, records label provenance, and applies thepromotion margin:```bashpython -m src.models.train --source offline```